In [ ]:
# SETUP: which run to read, output folders, plot style, helpers
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

RUN_ID = "run_20261114T211752Z"      # the final analysis run these figures come from
WORKBOOK = os.path.join("analysis_results", RUN_ID, "results_all_tables.xlsx")
OUT = "paper_outputs"
TAB_DIR, FIG_DIR = os.path.join(OUT, "tables"), os.path.join(OUT, "figures")
os.makedirs(TAB_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)

X = pd.read_excel(WORKBOOK, sheet_name=None)          # every sheet, as {name: table}
print("read", WORKBOOK, "|", len(X), "sheets |", X["README"].iloc[0]["description"])

# a warning if the validation sheets were not based on the consensus
va = X["validation_agreement"].iloc[0]
if int(va["unresolved_pairs"]) > 0:
    print(f"WARNING: validation error rates use '{va['error_rates_based_on']}' "
          f"({int(va['unresolved_pairs'])} unresolved pairs). Rerun the analysis with the consensus file first.")

MODELS = {"claude": "Claude Sonnet 5.5", "gemini": "Gemini 3.8 Flash"}
EMOTIONS = ["neutral", "fear", "joy"]
EMO_NAME = {"all": "All", "neutral": "Neutral", "fear": "Fear", "joy": "Joy"}
MEASURE_NAME = {"VC": "VC", "SC": "SC Agreement", "DSE": "DSE (modified)",
                "DSE_baseline": "DSE (baseline)", "LabelEntropy": "Label entropy"}
COLORS = {"claude": "#D55E00", "gemini": "#0072B2"}                       # colour-blind safe
EMO_COLORS = {"neutral": "#555555", "fear": "#CC3311", "joy": "#EE9900"}

# IEEE style: serif font, 8 pt, single column 3.5 in wide, double column 7.16 in
plt.rcParams.update({"font.family": "serif", "font.size": 8, "axes.titlesize": 8, "axes.labelsize": 8,
                     "legend.fontsize": 7, "xtick.labelsize": 7, "ytick.labelsize": 7,
                     "axes.spines.top": False, "axes.spines.right": False, "savefig.bbox": "tight"})
COL1, COL2 = 3.5, 7.16

def save_fig(fig, name):
    """Save a figure as PDF (for LaTeX) and PNG (for Word/slides)."""
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"), dpi=300)
    print("saved figure", name)
    plt.show()

def tex_escape(s):
    s = str(s)
    for a, b in [("\\", r"\textbackslash{}"), ("&", r"\&"), ("%", r"\%"), ("_", r"\_"), ("#", r"\#"),
                 ("Δ", r"$\Delta$"), ("κ", r"$\kappa$"), ("ρ", r"$\rho$"), ("→", r"$\rightarrow$"), ("−", "$-$")]:
        s = s.replace(a, b)
    return s

PAPER_TABLES = {}          # every paper table, also written to one Excel file at the end

def save_table(df, name, caption, note=None, wide=False):
    """Save a table as an IEEE LaTeX table (caption above, booktabs rules) and keep it for the Excel file."""
    env = "table*" if wide else "table"
    cols = "l" * 1 + "c" * (df.shape[1] - 1)
    def head(c):                              # long headers wrap onto two lines
            text = tex_escape(c).replace(" (", r" \\ (")
            return r"\textbf{\begin{tabular}[c]{@{}c@{}}" + text + r"\end{tabular}}"
    lines = [rf"\begin{{{env}}}[htbp]", rf"\caption{{{tex_escape(caption)}}}", rf"\label{{tab:{name}}}",
             r"\centering", r"\scriptsize", r"\setlength{\tabcolsep}{3pt}",
             r"\begin{adjustbox}{max width=\linewidth}",       # shrinks the table only if it is too wide
             rf"\begin{{tabular}}{{{cols}}}", r"\toprule",
             " & ".join(head(c) for c in df.columns) + r" \\", r"\midrule"]
    for _, r in df.iterrows():
        lines.append(" & ".join("" if pd.isna(v) else tex_escape(v) for v in r.values) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{adjustbox}"]
    if note:
        lines += [r"\par\vspace{2pt}", rf"\parbox{{\linewidth}}{{\scriptsize {tex_escape(note)}}}"]
    lines.append(rf"\end{{{env}}}")
    with open(os.path.join(TAB_DIR, f"{name}.tex"), "w") as f:
        f.write("\n".join(lines) + "\n")
    PAPER_TABLES[name] = df
    print(f"saved table {name} ({len(df)} rows)")
    display(df) if "display" in globals() else print(df.to_string(index=False))

def ci(est, lo, hi, d=3):
    """Format an estimate with its 95% interval, e.g. 0.567 [0.457, 0.670]."""
    return f"{est:.{d}f} [{lo:.{d}f}, {hi:.{d}f}]".replace("-", "−")

def wide_metric(sheet, family, measures, emotions=("all", "neutral", "fear", "joy")):
    """Pull rows of a results sheet into long form: model, measure, emotion, est, lo, hi."""
    d = X[sheet]
    d = d[(d["family"] == family) & d["measure"].isin(measures) & d["emotion"].isin(emotions)]
    rows = []
    for _, r in d.iterrows():
        for m in MODELS:
            rows.append({"model": m, "measure": r["measure"], "emotion": r["emotion"],
                         "est": r[f"{m}_estimate"], "lo": r[f"{m}_ci_low"], "hi": r[f"{m}_ci_high"]})
    return pd.DataFrame(rows)

P = X["prompt_level_data"]
P = P[P["included"]].copy()                     # prompts with enough valid trials (all 1080 here)

In [ ]:
# Table: data quality
dq = X["data_quality"]
t = pd.DataFrame({
    "Model": dq["model"].map(MODELS),
    "Responses": dq["responses"], "Valid": dq["valid_responses"], "Unparseable": dq["unparseable_responses"],
    "Prompts": dq["prompts"], "Excluded prompts": dq["prompts_excluded_lt_min_valid"],
    "Ties": dq["ties_among_included"]})
save_table(t, "data_quality", "Data Quality per Model",
           note="Each prompt was run 11 times; prompts with fewer than 9 valid responses would be excluded. "
                "Ties are prompts without a single majority answer.")

In [ ]:
# Table: validation of the semantic grouping
er = X["validation_error_rates"].copy()
er["cell"] = er.apply(lambda r: f"{int(r['errors'])}/{int(r['pairs'])} ({r['error_rate']:.0%})", axis=1)
t = er.pivot_table(index=["model", "method"], columns="relation", values="cell", aggfunc="first").reset_index()
t["model"] = t["model"].map(MODELS); t["method"] = t["method"].str.capitalize()
t = t.rename(columns={"model": "Model", "method": "Grouping", "over-merge": "Over-merge", "over-split": "Over-split"})
t = t[["Model", "Grouping", "Over-split", "Over-merge"]]
save_table(t, "grouping_validation", "Manual Validation of Semantic Grouping",
           note=f"Errors/pairs rated (rate). {int(va['pairs_rated_by_both'])} pairs rated independently by two "
                f"reviewers; initial agreement {va['percent_agreement']:.1%}, Cohen's κ = {va['cohens_kappa']:.2f}. "
                f"Error rates based on: {va['error_rates_based_on']}.")

In [ ]:
# question-level bootstrap for the mean of each uncertainty measure (the workbook only has their SD)
# Same scheme as the analysis: resample questions within each gold class, 2000 draws, seed 2026.
B, SEED = 2000, 2026
rng = np.random.default_rng(SEED)
q_by_gold = {g: np.array(sorted(d["question_id"].unique())) for g, d in P.groupby("gold")}
draws = [np.concatenate([rng.choice(q, size=len(q), replace=True) for q in q_by_gold.values()]) for _ in range(B)]
MEANS = {"mean_vc": "VC", "sc": "SC Agreement", "dse_modified": "DSE (modified)"}

def boot_mean(d, col):
    by_q = d.groupby("question_id")[col].agg(["sum", "count"])
    s, n = by_q["sum"].reindex(by_q.index), by_q["count"]
    vals = [s.reindex(qs).sum() / n.reindex(qs).sum() for qs in draws]
    return d[col].mean(), *np.percentile(vals, [2.5, 97.5])

rows = []
for m in MODELS:
    for e in ["all"] + EMOTIONS:
        d = P[P["model"] == m] if e == "all" else P[(P["model"] == m) & (P["emotion"] == e)]
        for col, name in MEANS.items():
            est, lo, hi = boot_mean(d, col)
            rows.append({"model": m, "emotion": e, "measure": name, "est": est, "lo": lo, "hi": hi})
UNC = pd.DataFrame(rows)

In [ ]:
# Table: accuracy and mean uncertainty by emotion
acc = wide_metric("rq1_accuracy", "acc", ["prompt", "response"])
rows = []
for m in MODELS:
    for e in ["all"] + EMOTIONS:
        r = {"Model": MODELS[m], "Emotion": EMO_NAME[e]}
        for meas, lab in [("prompt", "Accuracy (prompt)"), ("response", "Accuracy (response)")]:
            a = acc[(acc.model == m) & (acc.emotion == e) & (acc.measure == meas)].iloc[0]
            r[lab] = ci(a.est, a.lo, a.hi)
        for name in MEANS.values():
            u = UNC[(UNC.model == m) & (UNC.emotion == e) & (UNC.measure == name)].iloc[0]
            r[f"Mean {name}"] = ci(u.est, u.lo, u.hi)
        rows.append(r)
save_table(pd.DataFrame(rows), "rq1_accuracy_uncertainty",
           "Accuracy and Mean Uncertainty by Emotional Framing (RQ1)", wide=True,
           note="Values are estimates with 95% question-level bootstrap intervals (2,000 resamples, stratified by "
                "gold label). VC = mean verbalized confidence (0-1); DSE in nats (higher = more uncertain).")

In [ ]:
# Figure: accuracy by emotion
fig, ax = plt.subplots(figsize=(COL1, 2.2))
a = acc[(acc.measure == "prompt") & (acc.emotion != "all")]
x = np.arange(len(EMOTIONS)); w = 0.3
for i, m in enumerate(MODELS):
    d = a[a.model == m].set_index("emotion").loc[EMOTIONS]
    ax.bar(x + (i - 0.5) * w, d.est, w, color=COLORS[m], label=MODELS[m],
           yerr=[d.est - d.lo, d.hi - d.est], capsize=2, error_kw={"lw": 0.8})
ax.set_xticks(x, [EMO_NAME[e] for e in EMOTIONS]); ax.set_ylim(0, 1)
ax.set_ylabel("Prompt-level accuracy"); ax.legend(frameon=False, loc="upper left", ncol=2)
save_fig(fig, "fig_rq1_accuracy")

In [ ]:
# Figure: mean uncertainty by emotion, one panel per measure
fig, axes = plt.subplots(1, 3, figsize=(COL2, 2.0))
for ax, name in zip(axes, MEANS.values()):
    for i, m in enumerate(MODELS):
        d = UNC[(UNC.model == m) & (UNC.measure == name)].set_index("emotion").loc[EMOTIONS]
        ax.errorbar(x + (i - 0.5) * 0.2, d.est, yerr=[d.est - d.lo, d.hi - d.est], fmt="o", ms=3,
                    capsize=2, lw=0.8, color=COLORS[m], label=MODELS[m])
    ax.set_xticks(x, [EMO_NAME[e] for e in EMOTIONS]); ax.set_title(name)
fig.legend(*axes[0].get_legend_handles_labels(), frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2)
save_fig(fig, "fig_rq1_uncertainty")

In [ ]:
# Table: Brier, ECE (VC) and AUROC (VC, SC, DSE) by emotion
cal = pd.concat([wide_metric("calibration_vc", f, ["vc"]).assign(metric=f) for f in ("brier", "ece")])
au = wide_metric("auroc", "auroc", ["VC", "SC", "DSE", "DSE_baseline"])
rows = []
for m in MODELS:
    for e in ["all"] + EMOTIONS:
        r = {"Model": MODELS[m], "Emotion": EMO_NAME[e]}
        for f, lab in [("brier", "Brier (VC)"), ("ece", "ECE (VC)")]:
            c = cal[(cal.model == m) & (cal.emotion == e) & (cal.metric == f)].iloc[0]
            r[lab] = ci(c.est, c.lo, c.hi)
        for meas in ["VC", "SC", "DSE", "DSE_baseline"]:
            c = au[(au.model == m) & (au.emotion == e) & (au.measure == meas)].iloc[0]
            r[f"AUROC {MEASURE_NAME[meas]}"] = ci(c.est, c.lo, c.hi)
        rows.append(r)
save_table(pd.DataFrame(rows), "rq2_calibration_auroc",
           "Calibration and Discrimination of Uncertainty Measures (RQ2)", wide=True,
           note="Brier and ECE (10 bins) use mean VC against prompt-level correctness (lower = better). AUROC: "
                "how well each measure separates correct from incorrect prompts (0.5 = chance); DSE is negated so "
                "higher = more confident. 95% question-level bootstrap intervals.")

In [ ]:
# Figure: reliability diagram for VC, one panel per model, one line per emotion
fig, axes = plt.subplots(1, 2, figsize=(COL2, 2.6), sharey=True)
edges = np.linspace(0, 1, 11)
for ax, m in zip(axes, MODELS):
    ax.plot([0, 1], [0, 1], ls="--", lw=0.8, color="grey", label="Perfect calibration")
    for e in EMOTIONS:
        d = P[(P.model == m) & (P.emotion == e)]
        b = np.clip(np.digitize(d.mean_vc, edges[1:-1], right=True), 0, 9)
        g = d.groupby(b).agg(conf=("mean_vc", "mean"), acc=("correct", "mean"), n=("correct", "size"))
        ax.plot(g.conf, g.acc, marker="o", ms=3, lw=1, color=EMO_COLORS[e], label=EMO_NAME[e])
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_title(MODELS[m]); ax.set_xlabel("Mean verbalized confidence")
axes[0].set_ylabel("Observed accuracy"); axes[0].legend(frameon=False, loc="upper left")
save_fig(fig, "fig_rq2_reliability")

In [ ]:
# Figure: ROC curves, one panel per model
fig, axes = plt.subplots(1, 2, figsize=(COL2, 2.6), sharey=True)
scores = {"VC": P.mean_vc, "SC": P.sc, "DSE": -P.dse_modified}       # higher = more confident
for ax, m in zip(axes, MODELS):
    d = P[P.model == m]
    for meas, s in scores.items():
        fpr, tpr, _ = roc_curve(d.correct, s[d.index])
        ax.plot(fpr, tpr, lw=1, label=f"{MEASURE_NAME[meas]} ({roc_auc_score(d.correct, s[d.index]):.2f})")
    ax.plot([0, 1], [0, 1], ls="--", lw=0.8, color="grey")
    ax.set_title(MODELS[m]); ax.set_xlabel("False positive rate"); ax.legend(frameon=False, loc="lower right")
axes[0].set_ylabel("True positive rate")
save_fig(fig, "fig_rq2_roc")

In [ ]:
# Table: ΔAUROC (Neutral → Fear, Neutral → Joy), plus Claude − Gemini
da = wide_metric("robustness_dauroc", "dauroc", ["VC", "SC", "DSE", "DSE_baseline"], emotions=("fear", "joy"))
md = X["model_difference"]; md = md[md.family == "dauroc"]
rows = []
for meas in ["VC", "SC", "DSE", "DSE_baseline"]:
    for e in ["fear", "joy"]:
        r = {"Measure": MEASURE_NAME[meas], "Change": f"Neutral → {EMO_NAME[e]}"}
        for m in MODELS:
            c = da[(da.model == m) & (da.measure == meas) & (da.emotion == e)].iloc[0]
            r[MODELS[m]] = ci(c.est, c.lo, c.hi)
        c = md[(md.measure == meas) & (md.emotion == e)].iloc[0]
        r["Claude − Gemini"] = ci(c.estimate, c.ci_low, c.ci_high)
        rows.append(r)
save_table(pd.DataFrame(rows), "rq3_dauroc", "Change in AUROC Under Emotional Framing (RQ3)", wide=True,
           note="ΔAUROC = AUROC under the emotion minus AUROC under Neutral; values near 0 mean the measure is "
                "robust. Intervals that exclude 0 indicate a change beyond question sampling variation.")

In [ ]:
# Figure: ΔAUROC dot plot
fig, ax = plt.subplots(figsize=(COL1, 2.8))
labels, y = [], 0
for meas in ["VC", "SC", "DSE", "DSE_baseline"]:
    for e in ["fear", "joy"]:
        for i, m in enumerate(MODELS):
            c = da[(da.model == m) & (da.measure == meas) & (da.emotion == e)].iloc[0]
            ax.errorbar(c.est, y + (i - 0.5) * 0.3, xerr=[[c.est - c.lo], [c.hi - c.est]], fmt="o", ms=3,
                        capsize=2, lw=0.8, color=COLORS[m], label=MODELS[m] if y == 0 else None)
        labels.append(f"{MEASURE_NAME[meas]}: {EMO_NAME[e]}"); y += 1
ax.axvline(0, color="grey", lw=0.8, ls="--")
ax.set_yticks(range(len(labels)), labels); ax.invert_yaxis()
ax.set_xlabel("ΔAUROC vs. Neutral")
ax.legend(frameon=False, loc="lower center", bbox_to_anchor=(0.4, 1.0), ncol=2)
save_fig(fig, "fig_rq3_dauroc")

In [ ]:
# Appendix tables: grouping comparison, example group sizes, thinking tokens, ties
gc = X["grouping_comparison"]
save_table(pd.DataFrame({"Model": gc.model.map(MODELS),
                         "Groups (labels)": gc.mean_clusters_label.round(2),
                         "Groups (baseline)": gc.mean_clusters_baseline.round(2),
                         "Groups (modified)": gc.mean_clusters_modified.round(2),
                         "DSE (labels)": gc.mean_dse_label.round(3),
                         "DSE (baseline)": gc.mean_dse_baseline.round(3),
                         "DSE (modified)": gc.mean_dse_modified.round(3),
                         "Mixed-label baseline groups": gc.prompts_with_mixed_label_baseline_cluster}),
           "app_grouping_comparison", "Mean Groups and DSE per Prompt by Grouping Method", wide=True)

gs = X["group_sizes"]
ex = pd.concat([gs[(gs.model == m) & (gs.n_groups_baseline > gs.n_groups_modified)].head(3) for m in MODELS] +
               [gs[(gs.model == m) & (gs.n_groups_modified > 1)].head(2) for m in MODELS])
save_table(pd.DataFrame({"Model": ex.model.map(MODELS), "Prompt": ex.key, "Responses": ex.n_responses,
                         "Answer labels": ex.labels, "Baseline groups": ex.groups_baseline,
                         "Modified groups": ex.groups_modified}),
           "app_group_size_examples", "Example Group Sizes per Prompt",
           note="Number of responses in each semantic group, largest first.")

tt = X["thinking_tokens"]
save_table(pd.DataFrame({"Emotion": tt.emotion.map(EMO_NAME), "Used thinking": tt.thought.map({True: "Yes", False: "No"}),
                         "Responses": tt.responses, "Accuracy": tt.accuracy.round(3),
                         "Mean confidence": tt.mean_confidence.round(1),
                         "Mean thinking tokens": tt.mean_thinking_tokens.round(0)}),
           "app_thinking_tokens", "Gemini Responses With and Without Thinking Tokens")

print(f"ties among included prompts: {len(X['ties'])} (no ties table needed if 0)")

In [ ]:
# Appendix tables: RQ1 breakdowns (gold class, template, answer confusion)
bg = X["by_gold_class"]; bg = bg[(bg.family == "acc") & (bg.measure == "prompt")]
rows = []
for m in MODELS:
    for g in ["yes", "no", "maybe"]:
        r = {"Model": MODELS[m], "Gold label": g.capitalize()}
        for e in ["all"] + EMOTIONS:
            c = bg[(bg.model == m) & (bg.gold == g) & (bg.emotion == e)].iloc[0]
            r[EMO_NAME[e]] = ci(c.estimate, c.ci_low, c.ci_high)
        rows.append(r)
save_table(pd.DataFrame(rows), "app_accuracy_by_gold", "Prompt-Level Accuracy by Gold Label", wide=True)

bt = X["by_template"]
save_table(pd.DataFrame({"Model": bt.model.map(MODELS), "Template": bt.template_id,
                         "Accuracy": bt.acc_prompt.round(3), "Brier (VC)": bt.brier_vc.round(3),
                         "ECE (VC)": bt.ece_vc.round(3), "AUROC VC": bt.auroc_VC.round(3),
                         "AUROC SC": bt.auroc_SC.round(3), "AUROC DSE": bt.auroc_DSE.round(3)}),
           "app_by_template", "Results by Prompt Template", wide=True)

ac = X["answer_confusion"]; ac = ac[ac.emotion == "all"]
save_table(pd.DataFrame({"Model": ac.model.map(MODELS), "Gold label": ac.gold.str.capitalize(),
                         "Answered Yes": ac.majority_yes, "Answered No": ac.majority_no,
                         "Answered Maybe": ac.majority_maybe}),
           "app_answer_confusion", "Majority Answer vs. Gold Label (All Emotions)")

In [ ]:
# Appendix table: calibration and AUROC by gold label (all emotions)
bg = X["by_gold_class"]; bg = bg[bg.emotion == "all"]
rows = []
for m in MODELS:
    for g in ["yes", "no", "maybe"]:
        d = bg[(bg.model == m) & (bg.gold == g)]
        get = lambda fam, meas: (lambda r: ci(r.estimate, r.ci_low, r.ci_high))(
            d[(d.family == fam) & (d.measure == meas)].iloc[0])
        rows.append({"Model": MODELS[m], "Gold label": g.capitalize(),
                     "Brier (VC)": get("brier", "vc"), "ECE (VC)": get("ece", "vc"),
                     "AUROC VC": get("auroc", "VC"), "AUROC SC Agreement": get("auroc", "SC"),
                     "AUROC DSE (modified)": get("auroc", "DSE")})
save_table(pd.DataFrame(rows), "app_calibration_by_gold",
           "Calibration and Discrimination by Gold Label (All Emotions)", wide=True,
           note="Brier and ECE (10 bins) use mean VC against prompt-level correctness (lower = better). "
                "AUROC: separation of correct from incorrect prompts (0.5 = chance); DSE (modified grouping) "
                "negated so higher = more confident. 95% question-level bootstrap intervals; 20 questions per gold label.")

In [ ]:
# Appendix tables: RQ2 extras (correlations, DSE beyond labels)
co = X["correlations"]
co = co[(co.emotion == "all") & co.measure_b.isin(["sc", "dse_modified"]) | ((co.emotion == "all") &
        (co.measure_a == "sc") & (co.measure_b == "dse_modified"))].drop_duplicates()
NM = {"mean_vc": "VC", "sc": "SC", "dse_modified": "DSE (modified)", "dse_baseline": "DSE (baseline)"}
co = co[co.measure_a.isin(NM) & co.measure_b.isin(NM)]
save_table(pd.DataFrame({"Model": co.model.map(MODELS), "Measure A": co.measure_a.map(NM),
                         "Measure B": co.measure_b.map(NM), "Spearman ρ": co.spearman_rho.round(3),
                         "p": co.p_value.map(lambda p: "<0.001" if p < 0.001 else f"{p:.3f}")}),
           "app_correlations", "Spearman Correlations Between Uncertainty Measures (All Emotions)")

db = X["dse_beyond_label"]
save_table(pd.DataFrame({"Model": db.model.map(MODELS), "Emotion": db.emotion.map(EMO_NAME),
                         "Prompts": db.n_prompts,
                         "DSE > label entropy": db.share_dse_above_label_entropy.map("{:.1%}".format),
                         "SC = 1 but DSE > 0": db.share_sc_1_but_dse_above_0.map("{:.1%}".format),
                         "Mean DSE − label entropy": db.mean_dse_minus_label_entropy.round(4)}),
           "app_dse_beyond_label", "How Often DSE Adds Information Beyond the Answer Label")

In [ ]:
# Appendix: RQ3 sensitivity (SBERT threshold, ties) + figure
ts = X["threshold_sensitivity"]
save_table(pd.DataFrame({"Threshold": ts.threshold.map(lambda t: f"{t:.2f}" + (" (primary)" if t == 0.85 else "")),
                         "Model": ts.model.map(MODELS), "Mean groups": ts.mean_clusters.round(3),
                         "Mean DSE": ts.mean_dse.round(3), "AUROC DSE": ts.auroc_DSE_all.round(3),
                         "ΔAUROC Fear": ts.dauroc_DSE_fear.round(3), "ΔAUROC Joy": ts.dauroc_DSE_joy.round(3)}),
           "app_threshold_sensitivity", "Sensitivity of DSE to the SBERT Threshold", wide=True)

fig, axes = plt.subplots(1, 2, figsize=(COL2, 2.2))
for m in MODELS:
    d = ts[ts.model == m]
    axes[0].plot(d.threshold, d.auroc_DSE_all, marker="o", ms=3, color=COLORS[m], label=MODELS[m])
    axes[1].plot(d.threshold, d.dauroc_DSE_fear, marker="o", ms=3, color=COLORS[m], label=f"{MODELS[m]}: Fear")
    axes[1].plot(d.threshold, d.dauroc_DSE_joy, marker="s", ms=3, ls="--", color=COLORS[m], label=f"{MODELS[m]}: Joy")
for ax in axes: ax.axvline(0.85, color="grey", lw=0.8, ls=":"); ax.set_xlabel("SBERT threshold")
axes[0].set_ylabel("AUROC (DSE)"); axes[1].set_ylabel("ΔAUROC (DSE)"); axes[1].axhline(0, color="grey", lw=0.8)
axes[0].legend(frameon=False); axes[1].legend(frameon=False, fontsize=6)
save_fig(fig, "fig_app_threshold_sensitivity")

n_ties = int(X["data_quality"]["ties_among_included"].sum())
print(f"tie sensitivity: {n_ties} ties, so counting ties as wrong "
      f"{'gives identical results (report in one sentence, no table)' if n_ties == 0 else 'changes results: see tie_sensitivity sheet'}")

In [ ]:
# Save every paper table in one Excel file and list the outputs
with pd.ExcelWriter(os.path.join(OUT, "paper_tables.xlsx"), engine="openpyxl") as xl:
    for name, df in PAPER_TABLES.items():
        df.to_excel(xl, sheet_name=name[:31], index=False)
print(f"{len(PAPER_TABLES)} tables -> {TAB_DIR}/*.tex and {OUT}/paper_tables.xlsx")
print("figures:", sorted(f for f in os.listdir(FIG_DIR) if f.endswith(".pdf")))